# USDOT Intersection Safety Challenge — Stage-1B
## Notebook 02 — Parser Validation & Data Dictionary

**Goal:** Establish exactly how every file type must be parsed before any EDA.  
**Key findings going in:**
- `ISC_all_timing.csv` and `v2xhub_timing.csv` are **headerless** — pandas misreads them
- `V2XHubSensor.csv` is **headerless** — 3 packed columns: `timestamp_ms | active_ids | event_list`
- GT is **wide-format** with class-prefixed column triplets
- Radar sensor JSON is a **list of MQTT-style records** with nested `payload.objects`
- Traffic-triggers JSON has **nested `payload.trigger_outputs`** per timestamp
- **5-hour timezone offset** exists between GT (UTC) and camera timestamps (local)

---
## 0. Setup

In [1]:
# ── Download zip (skip if already present from a previous notebook in this session)
import os
from pathlib import Path

DOWNLOAD_URL = 'https://data.transportation.gov/download/j58q-ymi4/application%2Fx-zip-compressed'
ZIP_PATH     = '/content/isc_stage1b.zip'

if Path(ZIP_PATH).exists():
    print(f'Zip already present ({Path(ZIP_PATH).stat().st_size/1e9:.2f} GB) — skipping download.')
else:
    print('Downloading ~30 GB zip...')
    ret = os.system(f'wget -q --show-progress -O "{ZIP_PATH}" "{DOWNLOAD_URL}"')
    if ret != 0:
        raise RuntimeError('wget failed. Check connection and re-run.')
    print(f'Done: {ZIP_PATH}')

Done: /content/isc_stage1b.zip


In [2]:
# ── Selective extraction — skip .pcap and video (same as notebooks 00 & 01)
import zipfile, re
from pathlib import Path
from collections import Counter

EXTRACT_ROOT    = Path('/content/isc_stage1b')
SKIP_EXTENSIONS = {'.pcap', '.mp4', '.avi', '.mkv', '.mov', '.bag', '.bin'}

if EXTRACT_ROOT.exists() and any(EXTRACT_ROOT.iterdir()):
    n = sum(1 for _ in EXTRACT_ROOT.rglob('*'))
    print(f'Already extracted ({n:,} items). Skipping.')
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        all_entries = zf.namelist()
        info_map    = {i.filename: i for i in zf.infolist()}
    to_extract = [e for e in all_entries
                  if not e.endswith('/') and Path(e).suffix.lower() not in SKIP_EXTENSIONS]
    skipped    = [e for e in all_entries
                  if not e.endswith('/') and Path(e).suffix.lower() in SKIP_EXTENSIONS]
    ext_mb  = sum(info_map[e].file_size for e in to_extract if e in info_map) / 1e6
    skip_gb = sum(info_map[e].file_size for e in skipped    if e in info_map) / 1e9
    print(f'To extract: {len(to_extract):,} files (~{ext_mb:.0f} MB)')
    print(f'Skipping  : {len(skipped):,} files (~{skip_gb:.1f} GB)  '
          f'{dict(Counter(Path(e).suffix.lower() for e in skipped))}')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for i, e in enumerate(to_extract):
            zf.extract(e, path=str(EXTRACT_ROOT))
            if (i+1) % 200 == 0 or (i+1) == len(to_extract):
                print(f'  {i+1:,}/{len(to_extract):,}', end='\r')
    print(f'\nExtracted to {EXTRACT_ROOT}')

To extract: 780 files (~248 MB)
Skipping  : 645 files (~32.0 GB)  {'.mp4': 533, '.pcap': 112}
  780/780
Extracted to /content/isc_stage1b


In [3]:
# ── Imports & config ───────────────────────────────────────────────────────
import json, re, warnings
from pathlib import Path
from collections import defaultdict

import numpy  as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 55)

DATA_ROOT = Path('/content/isc_stage1b')
OUT_DIR   = Path('/content/outputs/tables')
MAX_RUNS  = 3

OUT_DIR.mkdir(parents=True, exist_ok=True)

run_dirs = sorted(
    [p for p in DATA_ROOT.iterdir() if p.is_dir() and re.match(r'run_?\d+', p.name, re.I)]
)
selected = run_dirs[:MAX_RUNS]

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'Total runs: {len(run_dirs)}  |  Selected: {[r.name for r in selected]}')
print(f'Output dir: {OUT_DIR}')

DATA_ROOT : /content/isc_stage1b
Total runs: 41  |  Selected: ['Run_1003', 'Run_1023', 'Run_1027']
Output dir: /content/outputs/tables


In [4]:
# ── Shared helpers ─────────────────────────────────────────────────────────

def find_files(run_dir: Path, patterns: list) -> list:
    return [f for f in sorted(run_dir.iterdir())
            if f.is_file() and any(re.search(p, f.name, re.I) for p in patterns)]


def fmt_size(path: Path) -> str:
    s = path.stat().st_size
    return f'{s/1e6:.2f} MB' if s >= 1e6 else f'{s/1e3:.1f} KB' if s >= 1000 else f'{s} B'


def divider(title: str = '', width: int = 68, char: str = '─'):
    if title:
        pad = width - len(title) - 4
        print(f'\n  {char*2} {title} {char * max(0, pad)}')
    else:
        print('  ' + char * width)


def show_df(df: pd.DataFrame, n: int = 5, label: str = ''):
    if label:
        divider(label)
    print(df.head(n).to_string(index=False))


def parse_isc_ts(ts: str) -> float:
    """YYYY-MM-DD-HH-MM-SS_microseconds  →  Unix float seconds."""
    try:
        main, us = str(ts).rsplit('_', 1)
        return pd.to_datetime(main, format='%Y-%m-%d-%H-%M-%S').timestamp() + int(us)/1e6
    except Exception:
        return float('nan')


print('Helpers ready.')

Helpers ready.


---
## A. Parser Strategy Registry

Defines the parsing rules for every file type **before** touching any file.  
Each entry is a dict that will later feed the data dictionary.

In [5]:
# ── A. Parser strategy registry ───────────────────────────────────────────
# Each entry covers one file pattern.
# Fields:
#   file_pattern   : glob-style description
#   has_header     : True / False / 'uncertain'
#   delimiter      : ',' / 'auto' / None (JSON)
#   row_unit       : what one row represents
#   format_class   : wide / long / event_log / nested_json
#   ts_field       : column/key holding the primary timestamp
#   ts_format      : description of the timestamp format
#   assigned_cols  : list of column names to assign when has_header=False
#   notes          : key caveats

PARSER_REGISTRY = [
    {
        'file_pattern'  : 'Run_XXXX_GT.csv',
        'pattern_regex' : r'^Run_\d+_GT\.csv$',
        'has_header'    : True,
        'delimiter'     : ',',
        'row_unit'      : 'one timestamp (frame), many object columns',
        'format_class'  : 'wide',
        'ts_field'      : 'Time',
        'ts_format'     : 'Unix epoch float (seconds, UTC)',
        'assigned_cols' : [],
        'notes'         : ('Wide format. Columns are CLASS_FIELD triplets: '
                           'e.g. Passenger_Vehicle_xctr/yctr/zctr/xlen/ylen/zlen/xrot/yrot/zrot. '
                           'Duplicate objects get .1, .2 suffixes. ~10 Hz. Many NaN per row.'),
    },
    {
        'file_pattern'  : 'ISC_Run_XXXX_ISC_all_timing.csv',
        'pattern_regex' : r'^ISC_Run_\d+_ISC_all_timing\.csv$',
        'has_header'    : False,
        'delimiter'     : ',',
        'row_unit'      : 'one sensor/stream entry (metadata row)',
        'format_class'  : 'event_log',
        'ts_field'      : 'start_time / end_time',
        'ts_format'     : 'datetime string YYYY-MM-DD HH:MM:SS.ffffff',
        'assigned_cols' : ['sensor_type', 'sensor_name', 'ip_address',
                           'filename', 'start_time', 'end_time', 'duration'],
        'notes'         : ('HEADERLESS — first row IS data. pandas wrongly uses it as header. '
                           'Each row = one sensor stream. Provides start/end/duration for every '
                           'camera, radar, and V2X stream in the run. ~13 rows per file.'),
    },
    {
        'file_pattern'  : 'ISC_Run_XXXX_v2xhub_timing.csv',
        'pattern_regex' : r'^ISC_Run_\d+_v2xhub_timing\.csv$',
        'has_header'    : False,
        'delimiter'     : ',',
        'row_unit'      : 'one V2X hub stream entry',
        'format_class'  : 'event_log',
        'ts_field'      : 'start_time / end_time',
        'ts_format'     : 'datetime string YYYY-MM-DD HH:MM:SS.ffffff',
        'assigned_cols' : ['sensor_type', 'sensor_name', 'ip_address',
                           'filename', 'start_time', 'end_time', 'duration'],
        'notes'         : ('HEADERLESS — same schema as ISC_all_timing. '
                           '~1-2 rows (v2xhub_sensors + v2xhub_spat). Very small (~276 B). '
                           'Only present in some runs.'),
    },
    {
        'file_pattern'  : 'VisualCameraX_Run_XXXX_frame-timing.csv',
        'pattern_regex' : r'^VisualCamera\d+_Run_\d+_frame-timing\.csv$',
        'has_header'    : True,
        'delimiter'     : ',',
        'row_unit'      : 'one video frame',
        'format_class'  : 'long',
        'ts_field'      : 'Timestamp',
        'ts_format'     : 'ISC string: YYYY-MM-DD-HH-MM-SS_microseconds (LOCAL time)',
        'assigned_cols' : [],
        'notes'         : ('Has header. Columns: Image_number (0-based frame index into .mp4), '
                           'Timestamp. 8 cameras per run. ~1900-4060 frames/camera. '
                           'Approx 30 fps. Timestamps are LOCAL time (not UTC).'),
    },
    {
        'file_pattern'  : 'ThermalCameraX_Run_XXXX_frame-timing.csv',
        'pattern_regex' : r'^ThermalCamera\d+_Run_\d+_frame-timing\.csv$',
        'has_header'    : True,
        'delimiter'     : ',',
        'row_unit'      : 'one thermal video frame',
        'format_class'  : 'long',
        'ts_field'      : 'Timestamp',
        'ts_format'     : 'ISC string: YYYY-MM-DD-HH-MM-SS_microseconds (LOCAL time)',
        'assigned_cols' : [],
        'notes'         : ('Identical schema to VisualCamera timing. '
                           '5 thermal cameras per run. Same LOCAL time issue.'),
    },
    {
        'file_pattern'  : 'V2XHubSensor_Run_XXXX.csv',
        'pattern_regex' : r'^V2XHubSensor_Run_\d+\.csv$',
        'has_header'    : False,
        'delimiter'     : ',',
        'row_unit'      : 'one 1-second SPaT/BSM snapshot',
        'format_class'  : 'event_log',
        'ts_field'      : 'timestamp_ms',
        'ts_format'     : 'Unix epoch milliseconds (integer)',
        'assigned_cols' : ['timestamp_ms', 'active_signal_ids', 'event_list'],
        'notes'         : ('HEADERLESS — 3 columns. Col 0: Unix ms timestamp. '
                           'Col 1: active signal/phase IDs (list-like string e.g. "[2, 8]"). '
                           'Col 2: event list (often empty "[]"). '
                           'Only present in runs with V2X hardware. ~70 rows per file.'),
    },
    {
        'file_pattern'  : 'Radars_Run_XXXX_sensorN.json',
        'pattern_regex' : r'^Radars_Run_\d+_sensor\d+\.json$',
        'has_header'    : None,
        'delimiter'     : None,
        'row_unit'      : 'one MQTT message (per radar cycle ~0.1 s)',
        'format_class'  : 'nested_json',
        'ts_field'      : 'payload.timestamp',
        'ts_format'     : 'ISO 8601 UTC: YYYY-MM-DDTHH:MM:SS+00:00',
        'assigned_cols' : [],
        'notes'         : ('Top-level list of dicts. Keys per record: '
                           'topic / payload / qos / receivedAt / retain. '
                           'payload.objects is a list of detected objects per cycle. '
                           'Object fields: id, class, heading, speed, length, '
                           'position_facing, position_front, closest_lane, within_zone, '
                           'acceleration, tracking_status. ~1900 records per sensor.'),
    },
    {
        'file_pattern'  : 'Radars_Run_XXXX_traffic-triggers-output.json',
        'pattern_regex' : r'^Radars_Run_\d+_traffic-triggers-output\.json$',
        'has_header'    : None,
        'delimiter'     : None,
        'row_unit'      : 'one MQTT trigger message (per timestamp)',
        'format_class'  : 'nested_json',
        'ts_field'      : 'payload.timestamp',
        'ts_format'     : 'ISO 8601 UTC: YYYY-MM-DDTHH:MM:SS+00:00',
        'assigned_cols' : [],
        'notes'         : ('Top-level list of ~2525 dicts. '
                           'payload.trigger_outputs[].traffic_triggers[] contains: '
                           'associated_lane, associated_sensor, associated_zone, reference_name. '
                           'Key file for conflict/zone/lane trigger analysis. 6-9 MB per run.'),
    },
]

# Build lookup: regex → strategy
PARSER_LOOKUP = {e['pattern_regex']: e for e in PARSER_REGISTRY}

print(f'Parser registry: {len(PARSER_REGISTRY)} file types defined.\n')
for e in PARSER_REGISTRY:
    hdr = {True: 'YES', False: 'NO ', None: 'N/A'}[e['has_header']]
    print(f'  header={hdr}  fmt={e["format_class"]:12s}  {e["file_pattern"]}')

Parser registry: 8 file types defined.

  header=YES  fmt=wide          Run_XXXX_GT.csv
  header=NO   fmt=event_log     ISC_Run_XXXX_ISC_all_timing.csv
  header=NO   fmt=event_log     ISC_Run_XXXX_v2xhub_timing.csv
  header=YES  fmt=long          VisualCameraX_Run_XXXX_frame-timing.csv
  header=YES  fmt=long          ThermalCameraX_Run_XXXX_frame-timing.csv
  header=NO   fmt=event_log     V2XHubSensor_Run_XXXX.csv
  header=N/A  fmt=nested_json   Radars_Run_XXXX_sensorN.json
  header=N/A  fmt=nested_json   Radars_Run_XXXX_traffic-triggers-output.json


---
## B. CSV Header Validation

For every CSV file type, show **wrong parse** (pandas default) side-by-side with  
**correct parse** (header=None + assigned column names).  
Focus on the three headerless files confirmed from notebook 01.

In [6]:
# ── B helper: side-by-side wrong vs correct parse ─────────────────────────
def compare_parse(path, assigned_cols, n_show=5):
    """
    Read a CSV twice (default header vs header=None) and print both.
    Robust to column-count mismatches: if the file has MORE columns than
    assigned_cols (e.g. an unquoted '[2, 8]' splits into two columns),
    extra columns are named col_N so we never raise a length error.
    """
    print(f'\n  File: {path.name}  ({fmt_size(path)})')

    df_wrong = pd.read_csv(path, low_memory=False)           # wrong
    df_right = pd.read_csv(path, header=None, low_memory=False)  # correct
    n_actual = df_right.shape[1]

    if assigned_cols:
        n_named    = len(assigned_cols)
        final_cols = list(assigned_cols[:n_actual]) + [
            f'col_{i}' for i in range(n_actual - n_named)
        ]
        df_right.columns = final_cols
        if n_actual != n_named:
            print(f'  [NOTE] Expected {n_named} cols but found {n_actual}.')
            print('         Extra cols named col_N — unquoted list values split by comma.')

    divider('WRONG parse (header=0, pandas default)')
    print(f'  Shape: {df_wrong.shape}  |  Columns: {list(df_wrong.columns)}')
    print(df_wrong.head(n_show).to_string(index=False))

    divider('CORRECT parse (header=None + assigned column names)')
    print(f'  Shape: {df_right.shape}  |  Columns: {list(df_right.columns)}')
    print(df_right.head(n_show).to_string(index=False))

    print()
    print(f'  >> Wrong parse lost 1 data row (used as header). '
          f'Correct parse has {df_right.shape[0]} data rows.')
    return df_right

print('Side-by-side comparison helper defined.')


Side-by-side comparison helper defined.


In [7]:
# ── B1. ISC_all_timing.csv — confirmed HEADERLESS ─────────────────────────
print('=' * 68)
print('  B1. ISC_Run_XXXX_ISC_all_timing.csv')
print('=' * 68)
print("""
  Evidence from notebook 01:
  pandas column names were: 'radar', 'Radars', '192.168.55.44',
  'Radars_S02C07R7_2.json', '2024-02-20 13:36:43.506564', ...
  Those are actual DATA values — the first sensor row got eaten as header.

  Correct column assignment:
  sensor_type | sensor_name | ip_address | filename | start_time | end_time | duration
""")

COLS_ISC = ['sensor_type', 'sensor_name', 'ip_address',
            'filename', 'start_time', 'end_time', 'duration']

isc_timing_parsed = {}
for run in selected:
    files = find_files(run, [r'^ISC_Run_\d+_ISC_all_timing\.csv$'])
    if not files:
        print(f'  {run.name}: not found')
        continue
    print(f'\n{"-"*68}')
    print(f'  Run: {run.name}')
    df = compare_parse(files[0], COLS_ISC)
    isc_timing_parsed[run.name] = df

    # Parse timestamps now that columns are correct
    for col in ['start_time', 'end_time']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    if 'start_time' in df.columns:
        divider('Parsed timestamp range')
        print(f'  start_time min: {df["start_time"].min()}')
        print(f'  end_time   max: {df["end_time"].max()}')
        print(f'  Unique sensor_types: {sorted(df["sensor_type"].dropna().unique())}')

  B1. ISC_Run_XXXX_ISC_all_timing.csv

  Evidence from notebook 01:
  pandas column names were: 'radar', 'Radars', '192.168.55.44',
  'Radars_S02C07R7_2.json', '2024-02-20 13:36:43.506564', ...
  Those are actual DATA values — the first sensor row got eaten as header.

  Correct column assignment:
  sensor_type | sensor_name | ip_address | filename | start_time | end_time | duration


--------------------------------------------------------------------
  Run: Run_1003

  File: ISC_Run_1003_ISC_all_timing.csv  (2.0 KB)

  ── WRONG parse (header=0, pandas default) ──────────────────────────
  Shape: (14, 7)  |  Columns: ['radar_camera', 'VisualCamera5', '192.168.55.44:51554', 'VisualCamera5_S06C06R06.mp4', '2023-11-20 14:53:02.814874', '2023-11-20 14:54:16.085819', '0:01:13.234703']
radar_camera  VisualCamera5 192.168.55.44:51554  VisualCamera5_S06C06R06.mp4 2023-11-20 14:53:02.814874 2023-11-20 14:54:16.085819 0:01:13.234703
        flir ThermalCamera2      192.168.55.181 ThermalCamera2

In [8]:
# ── B2. v2xhub_timing.csv — confirmed HEADERLESS ──────────────────────────
print('=' * 68)
print('  B2. ISC_Run_XXXX_v2xhub_timing.csv')
print('=' * 68)
print("""
  Evidence from notebook 01:
  pandas column names were: 'v2xhub_sensors', 'V2XHubSensor',
  '192.168.55.149', 'V2XHubSensor_S02C07R7_2.csv', ...
  Same pattern as ISC_all_timing — first data row used as header.
  Same 7-column schema.
""")

v2xhub_timing_parsed = {}
for run in selected:
    files = find_files(run, [r'^ISC_Run_\d+_v2xhub_timing\.csv$'])
    if not files:
        print(f'  {run.name}: not present (V2X not deployed in this run)')
        continue
    print(f'\n{"-"*68}')
    print(f'  Run: {run.name}')
    df = compare_parse(files[0], COLS_ISC)
    v2xhub_timing_parsed[run.name] = df
    print(f'  Unique sensor_types: {df["sensor_type"].dropna().unique().tolist()}')

  B2. ISC_Run_XXXX_v2xhub_timing.csv

  Evidence from notebook 01:
  pandas column names were: 'v2xhub_sensors', 'V2XHubSensor',
  '192.168.55.149', 'V2XHubSensor_S02C07R7_2.csv', ...
  Same pattern as ISC_all_timing — first data row used as header.
  Same 7-column schema.

  Run_1003: not present (V2X not deployed in this run)

--------------------------------------------------------------------
  Run: Run_1023

  File: ISC_Run_1023_v2xhub_timing.csv  (276 B)

  ── WRONG parse (header=0, pandas default) ──────────────────────────
  Shape: (1, 7)  |  Columns: ['v2xhub_sensors', 'V2XHubSensor', '192.168.55.149', 'V2XHubSensor_S07C20R7_1.csv', '2024-02-22 20:15:25.866579', '2024-02-22 20:16:29.942382', '0:01:04.075803']
v2xhub_sensors V2XHubSensor 192.168.55.149 V2XHubSensor_S07C20R7_1.csv 2024-02-22 20:15:25.866579 2024-02-22 20:16:29.942382 0:01:04.075803
   v2xhub_spat   V2XHubSPaT 192.168.55.149  V2XHubSPaT_S07C20R7_1.pcap 2024-02-22 20:15:25.947258 2024-02-22 20:16:29.340559 0:01:03

In [9]:
# ── B3. V2XHubSensor_Run_XXXX.csv — confirmed HEADERLESS ─────────────────
print('=' * 68)
print('  B3. V2XHubSensor_Run_XXXX.csv')
print('=' * 68)
print('''
  Root cause of the original ValueError:
  The value "[2, 8]" is NOT quoted in the CSV.
  pandas splits it on the comma into two columns: "[2" and " 8]".
  So the file actually has 4 raw columns, not 3.

  Fix:
    1. Read with header=None  -> 4 columns
    2. Merge cols 1..(n-2) back into one active_signal_ids string
    3. Assign:  timestamp_ms | active_signal_ids | event_list
''')

COLS_V2X = ['timestamp_ms', 'active_signal_ids', 'event_list']

v2x_sensor_parsed = {}

for run in selected:
    files = find_files(run, [r'V2XHubSensor_Run_\d+\.csv'])
    if not files:
        print(f'  {run.name}: not present')
        continue

    fpath = files[0]
    print(f'\n{"-"*68}')
    print(f'  Run: {run.name}')
    print(f'  File: {fpath.name}  ({fmt_size(fpath)})')

    # ── Wrong parse (shows the column-split problem clearly) ──────────
    df_wrong = pd.read_csv(fpath, low_memory=False)
    divider('WRONG parse (header=0, pandas default)')
    print(f'  Shape: {df_wrong.shape}  |  Columns: {list(df_wrong.columns)}')
    print(df_wrong.head(5).to_string(index=False))

    # ── Correct parse ─────────────────────────────────────────────────
    df_raw = pd.read_csv(fpath, header=None, low_memory=False)
    n_cols = df_raw.shape[1]
    print(f'\n  [INFO] Raw column count (header=None): {n_cols}')

    if n_cols == 3:
        # Already 3 cols — list values are quoted in this file
        df_raw.columns = COLS_V2X
        df_fix = df_raw.copy()
    elif n_cols >= 4:
        # Unquoted list caused extra split — merge middle columns
        # Layout: col0=timestamp | col1..col(n-2)=parts of list | col(n-1)=event_list
        df_fix = pd.DataFrame()
        df_fix['timestamp_ms']      = df_raw.iloc[:, 0]
        df_fix['active_signal_ids'] = (
            df_raw.iloc[:, 1:n_cols - 1]
                  .astype(str)
                  .apply(','.join, axis=1)
                  .str.strip()
        )
        df_fix['event_list'] = df_raw.iloc[:, n_cols - 1]
    else:
        print(f'  [WARN] Unexpected column count {n_cols} — skipping')
        continue

    v2x_sensor_parsed[run.name] = df_fix

    divider('CORRECT parse (header=None + list-column merge)')
    print(f'  Shape: {df_fix.shape}  |  Columns: {list(df_fix.columns)}')
    print(df_fix.head(5).to_string(index=False))

    print()
    print(f'  >> Wrong parse lost 1 data row. Correct parse has {len(df_fix)} rows.')

    # ── Decode timestamps ──────────────────────────────────────────────
    divider('Timestamp parsed')
    ts = pd.to_numeric(df_fix['timestamp_ms'], errors='coerce')
    print(f'  Raw    : min={ts.min()}  max={ts.max()}')
    print(f'  Seconds: {ts.min()/1000:.3f}  ->  {ts.max()/1000:.3f}')
    print(f'  UTC    : {pd.to_datetime(ts.min(), unit="ms")}'
          f'  ->  {pd.to_datetime(ts.max(), unit="ms")}')
    print(f'  Rows   : {len(df_fix)}  (~{len(df_fix)} seconds of V2X data)')

    divider('active_signal_ids unique values')
    print(f'  {df_fix["active_signal_ids"].unique().tolist()}')
    print('  -> list of currently-active signal phase IDs')

    divider('event_list unique values')
    print(f'  {df_fix["event_list"].unique().tolist()}')
    print('  -> SPaT events fired that second (empty = no phase-change)')


  B3. V2XHubSensor_Run_XXXX.csv

  Root cause of the original ValueError:
  The value "[2, 8]" is NOT quoted in the CSV.
  pandas splits it on the comma into two columns: "[2" and " 8]".
  So the file actually has 4 raw columns, not 3.

  Fix:
    1. Read with header=None  -> 4 columns
    2. Merge cols 1..(n-2) back into one active_signal_ids string
    3. Assign:  timestamp_ms | active_signal_ids | event_list

  Run_1003: not present

--------------------------------------------------------------------
  Run: Run_1023
  File: V2XHubSensor_Run_1023.csv  (1.7 KB)

  ── WRONG parse (header=0, pandas default) ──────────────────────────
  Shape: (64, 4)  |  Columns: ['1708650925643', ' [2', ' 8]', ' []']
 1708650925643  [2  8]  []
 1708650926643  [2  8]  []
 1708650927643  [2  8]  []
 1708650928642  [2  8]  []
 1708650929642  [2  8]  []
 1708650930642  [2  8]  []

  [INFO] Raw column count (header=None): 4

  ── CORRECT parse (header=None + list-column merge) ─────────────────
  Shape: (6

In [10]:
# ── B4. Confirm files that DO have correct headers ─────────────────────────
print('=' * 68)
print('  B4. Files with correct headers (no fix needed)')
print('=' * 68)

HAS_HEADER_TYPES = [
    ('GT CSV',               r'^Run_\d+_GT\.csv$'),
    ('Visual frame-timing',  r'^VisualCamera\d+_Run_\d+_frame-timing\.csv$'),
    ('Thermal frame-timing', r'^ThermalCamera\d+_Run_\d+_frame-timing\.csv$'),
]

run = selected[0]  # just use first run
for label, pattern in HAS_HEADER_TYPES:
    files = find_files(run, [pattern])
    if not files:
        print(f'  {label}: not found in {run.name}')
        continue
    f = files[0]
    df = pd.read_csv(f, nrows=3, low_memory=False)
    print(f'\n  {label}  ({f.name})')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Row 0  : {df.iloc[0].tolist()[:6]}...')
    print(f'  Verdict: header IS correct  (column names are not data values)')

  B4. Files with correct headers (no fix needed)

  GT CSV  (Run_1003_GT.csv)
  Columns: ['Time', 'Passenger_Vehicle_xctr', 'Passenger_Vehicle_yctr', 'Passenger_Vehicle_zctr', 'Passenger_Vehicle_xlen', 'Passenger_Vehicle_ylen', 'Passenger_Vehicle_zlen', 'Passenger_Vehicle_xrot', 'Passenger_Vehicle_yrot', 'Passenger_Vehicle_zrot', 'VRU_Child_xctr', 'VRU_Child_yctr', 'VRU_Child_zctr', 'VRU_Child_xlen', 'VRU_Child_ylen', 'VRU_Child_zlen', 'VRU_Child_xrot', 'VRU_Child_yrot', 'VRU_Child_zrot', 'VRU_Adult_Using_Walker_xctr', 'VRU_Adult_Using_Walker_yctr', 'VRU_Adult_Using_Walker_zctr', 'VRU_Adult_Using_Walker_xlen', 'VRU_Adult_Using_Walker_ylen', 'VRU_Adult_Using_Walker_zlen', 'VRU_Adult_Using_Walker_xrot', 'VRU_Adult_Using_Walker_yrot', 'VRU_Adult_Using_Walker_zrot']
  Row 0  : [1700509980.881572, nan, nan, nan, nan, nan]...
  Verdict: header IS correct  (column names are not data values)

  Visual frame-timing  (VisualCamera1_Run_1003_frame-timing.csv)
  Columns: ['Image_number', 'Timestam

---
## C. Special Handling — GT CSV (Wide Format)

The GT file is wide-format: each row is one timestamp, columns encode  
`CLASS_FIELD` pairs. Understanding the column naming scheme is essential  
before any EDA.

In [11]:
# ── C1. Load GT and decode column naming scheme ────────────────────────────
run       = selected[0]
gt_file   = find_files(run, [r'^Run_\d+_GT\.csv$'])[0]
gt_df     = pd.read_csv(gt_file, low_memory=False)

print(f'GT file : {gt_file.name}  ({fmt_size(gt_file)})')
print(f'Shape   : {gt_df.shape[0]} rows x {gt_df.shape[1]} columns')
print(f'Timestamp field: Time  (Unix epoch float, UTC)')

# ── Decode class prefixes ─────────────────────────────────────────────────
# Column pattern: {ObjectClass}_{FieldSuffix}
# Known suffixes: xctr yctr zctr (centroid x/y/z)
#                 xlen ylen zlen (bounding box dimensions)
#                 xrot yrot zrot (rotation angles)
# Duplicate instances: Passenger_Vehicle_xctr, Passenger_Vehicle_xctr.1, .2 ...

FIELD_SUFFIXES = ['xctr','yctr','zctr','xlen','ylen','zlen','xrot','yrot','zrot']

non_ts_cols = [c for c in gt_df.columns if c != 'Time']

# Extract unique base class names (strip .N suffix and known field suffixes)
classes = set()
for col in non_ts_cols:
    base = re.sub(r'\.\d+$', '', col)          # strip .1 .2
    for suf in FIELD_SUFFIXES:
        if base.endswith('_' + suf):
            cls = base[: -(len(suf) + 1)]
            classes.add(cls)
            break

divider('Unique object classes in GT columns')
for cls in sorted(classes):
    # Count how many instance slots exist for this class
    xctr_cols = [c for c in gt_df.columns if re.match(rf'^{re.escape(cls)}_xctr(\.\d+)?$', c)]
    n_slots   = len(xctr_cols)
    non_nan   = gt_df[xctr_cols].notna().any(axis=1).sum()
    print(f'  {cls:<40s}  slots={n_slots}  rows_with_data={non_nan}')

divider('Field suffixes and their meaning')
meaning = {
    'xctr/yctr/zctr': 'centroid position (x, y, z)',
    'xlen/ylen/zlen': 'bounding box dimensions (length along x, y, z)',
    'xrot/yrot/zrot': 'rotation angles around x, y, z axes',
}
for k, v in meaning.items():
    print(f'  _{k}  →  {v}')

divider('Duplicate instance columns (.1, .2 pattern)')
dup_cols = [c for c in gt_df.columns if re.search(r'\.\d+$', c)]
print(f'  Total duplicate-instance columns: {len(dup_cols)}')
print(f'  Examples: {dup_cols[:8]}')
print(f'\n  Meaning: when multiple objects of the same class exist in one frame,')
print(f'  extra instance columns are added as ClassName_Field, ClassName_Field.1, etc.')

divider('Timestamp (Time column)')
ts = pd.to_numeric(gt_df['Time'], errors='coerce')
print(f'  min: {ts.min():.3f}  →  {pd.to_datetime(ts.min(), unit="s")}')
print(f'  max: {ts.max():.3f}  →  {pd.to_datetime(ts.max(), unit="s")}')
print(f'  rows: {len(ts)}  approx rate: {round(len(ts)/(ts.max()-ts.min()), 2)} Hz')
print(f'  Timezone: UTC (Unix epoch)')

GT file : Run_1003_GT.csv  (171.9 KB)
Shape   : 758 rows x 28 columns
Timestamp field: Time  (Unix epoch float, UTC)

  ── Unique object classes in GT columns ─────────────────────────────
  Passenger_Vehicle                         slots=1  rows_with_data=223
  VRU_Adult_Using_Walker                    slots=1  rows_with_data=758
  VRU_Child                                 slots=1  rows_with_data=758

  ── Field suffixes and their meaning ────────────────────────────────
  _xctr/yctr/zctr  →  centroid position (x, y, z)
  _xlen/ylen/zlen  →  bounding box dimensions (length along x, y, z)
  _xrot/yrot/zrot  →  rotation angles around x, y, z axes

  ── Duplicate instance columns (.1, .2 pattern) ─────────────────────
  Total duplicate-instance columns: 0
  Examples: []

  Meaning: when multiple objects of the same class exist in one frame,
  extra instance columns are added as ClassName_Field, ClassName_Field.1, etc.

  ── Timestamp (Time column) ────────────────────────────────────────

In [12]:
# ── C2. Demo: melt ONE class from wide to long for downstream use ──────────
# This is NOT done during EDA — shown here as proof-of-concept for the parser.

# Pick the first class with the most data
best_cls, best_n = None, 0
for cls in sorted(classes):
    xctr_cols = [c for c in gt_df.columns if re.match(rf'^{re.escape(cls)}_xctr(\.\d+)?$', c)]
    n = gt_df[xctr_cols].notna().any(axis=1).sum()
    if n > best_n:
        best_n, best_cls = n, cls

print(f'Demo class: {best_cls}  ({best_n} non-empty frames)')

# Find all instance slot columns for this class
slot_pattern = rf'^{re.escape(best_cls)}_({'|'.join(FIELD_SUFFIXES)})(\.(\d+))?$'
cls_cols = [c for c in gt_df.columns if re.match(slot_pattern, c)]

# How many slots?
max_slot = max(
    (int(m.group(3)) if m.group(3) else 0)
    for c in cls_cols
    if (m := re.match(slot_pattern, c))
) + 1
print(f'Max simultaneous instances: {max_slot}')

# Build one long-format mini-DataFrame for slot 0 only (as demonstration)
slot0_cols = {}
for suf in FIELD_SUFFIXES:
    cname = f'{best_cls}_{suf}'
    if cname in gt_df.columns:
        slot0_cols[suf] = cname

demo = gt_df[['Time'] + list(slot0_cols.values())].copy()
demo.columns = ['Time'] + list(slot0_cols.keys())
demo = demo.dropna(subset=['xctr']).reset_index(drop=True)

divider(f'Slot-0 records for {best_cls} (first 5 rows)')
print(demo.head(5).to_string(index=False))
print(f'\n  Total non-NaN rows for slot 0: {len(demo)}')
print(f'  Parser note: to use GT data, iterate over slots and melt each into long format.')

Demo class: VRU_Adult_Using_Walker  (758 non-empty frames)
Max simultaneous instances: 1

  ── Slot-0 records for VRU_Adult_Using_Walker (first 5 rows) ────────
        Time      xctr      yctr      zctr     xlen     ylen     zlen  xrot  yrot  zrot
1.700510e+09 12.138721 -1.800893 -3.828306 1.110007 1.260583 1.838113     0     0 -90.0
1.700510e+09 12.138721 -1.800893 -3.828306 1.110007 1.260583 1.838113     0     0 -90.0
1.700510e+09 12.138721 -1.800893 -3.828306 1.110007 1.260583 1.838113     0     0 -90.0
1.700510e+09 12.138721 -1.800893 -3.828306 1.110007 1.260583 1.838113     0     0 -90.0
1.700510e+09 12.138721 -1.800893 -3.828306 1.110007 1.260583 1.838113     0     0 -90.0

  Total non-NaN rows for slot 0: 758
  Parser note: to use GT data, iterate over slots and melt each into long format.


---
## D. Special Handling — Camera Timing, ISC Timing, V2X, Radar JSONs

In [13]:
# ── D1. Camera frame-timing — confirm schema and parse ISC timestamps ───────
print('=' * 68)
print('  D1. Camera Frame-Timing CSV')
print('=' * 68)

run  = selected[0]
vis  = find_files(run, [r'^VisualCamera\d+_Run_\d+_frame-timing\.csv$'])
thm  = find_files(run, [r'^ThermalCamera\d+_Run_\d+_frame-timing\.csv$'])

cam_timing_results = {}
for label, files in [('Visual', vis), ('Thermal', thm)]:
    if not files:
        continue
    f  = files[0]
    df = pd.read_csv(f, low_memory=False)   # header=True confirmed

    divider(f'{label}: {f.name}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Shape  : {df.shape}')
    print(df.head(5).to_string(index=False))

    # Parse ISC timestamps
    ts_unix = df['Timestamp'].astype(str).apply(parse_isc_ts)
    valid   = ts_unix.dropna().pipe(lambda s: s[~np.isnan(s)])

    divider('Parsed ISC timestamps')
    t_min, t_max = valid.min(), valid.max()
    duration     = t_max - t_min
    n_frames     = len(valid)
    fps          = round(n_frames / duration, 2) if duration > 0 else float('nan')

    print(f'  raw sample   : {df["Timestamp"].iloc[0]}')
    print(f'  parsed min   : {pd.to_datetime(t_min, unit="s")}  (LOCAL TIME, not UTC)')
    print(f'  parsed max   : {pd.to_datetime(t_max, unit="s")}')
    print(f'  duration     : {duration:.2f} s')
    print(f'  frames       : {n_frames}')
    print(f'  approx fps   : {fps}')
    print(f'  Image_number : {int(df["Image_number"].min())} → {int(df["Image_number"].max())}')
    print(f'  Non-consec.  : {int((df["Image_number"].diff().dropna() != 1).sum())} gaps')

    cam_timing_results[label] = {'t_min': t_min, 't_max': t_max, 'fps': fps, 'n': n_frames}

print()
print('  *** TIMEZONE NOTE ***')
print('  Camera timestamps are LOCAL time (system clock).')
print('  GT timestamps are UTC (Unix epoch).')
print('  From notebook 01: GT shows 19:53 UTC, cameras show 14:53 local = 5 h behind UTC.')
print('  To align: convert camera local → UTC (determine UTC offset from run location).')

  D1. Camera Frame-Timing CSV

  ── Visual: VisualCamera1_Run_1003_frame-timing.csv ─────────────────
  Columns: ['Image_number', 'Timestamp']
  Shape  : (2247, 2)
 Image_number                  Timestamp
            0 2023-11-20-14-53-01_686968
            1 2023-11-20-14-53-01_718474
            2 2023-11-20-14-53-01_745849
            3 2023-11-20-14-53-01_767417
            4 2023-11-20-14-53-01_788887

  ── Parsed ISC timestamps ───────────────────────────────────────────
  raw sample   : 2023-11-20-14-53-01_686968
  parsed min   : 2023-11-20 14:53:01.686968088  (LOCAL TIME, not UTC)
  parsed max   : 2023-11-20 14:54:16.074453115
  duration     : 74.39 s
  frames       : 2247
  approx fps   : 30.21
  Image_number : 0 → 2246
  Non-consec.  : 0 gaps

  ── Thermal: ThermalCamera1_Run_1003_frame-timing.csv ───────────────
  Columns: ['Image_number', 'Timestamp']
  Shape  : (2249, 2)
 Image_number                  Timestamp
            0 2023-11-20-14-53-01_605550
            1 2023-11

In [14]:
# ── D2. ISC_all_timing.csv — correctly parsed version with full inspection ─
print('=' * 68)
print('  D2. ISC_all_timing.csv — correctly parsed')
print('=' * 68)

for run in selected:
    df = isc_timing_parsed.get(run.name)
    if df is None:
        print(f'  {run.name}: no data')
        continue

    divider(f'{run.name}  ({len(df)} sensor streams)')
    print(df.to_string(index=False))

    # Derived: per sensor_type summary
    if 'sensor_type' in df.columns:
        print(f'\n  sensor_type breakdown:')
        for stype, grp in df.groupby('sensor_type'):
            names = grp['sensor_name'].tolist()
            print(f'    {stype:<20s}  {len(grp)} entries  → {names[:4]}')

  D2. ISC_all_timing.csv — correctly parsed

  ── Run_1003  (15 sensor streams) ───────────────────────────────────
 sensor_type    sensor_name          ip_address                     filename                 start_time                   end_time       duration
radar_camera  VisualCamera5 192.168.55.44:51554  VisualCamera5_S06C06R06.mp4 2023-11-20 14:53:02.814874 2023-11-20 14:54:16.085819 0:01:13.234703
        flir ThermalCamera2      192.168.55.181 ThermalCamera2_S06C06R06.mp4 2023-11-20 14:53:01.609054 2023-11-20 14:54:16.100482 0:01:14.467779
radar_camera  VisualCamera7 192.168.55.44:53554  VisualCamera7_S06C06R06.mp4 2023-11-20 14:53:02.985637 2023-11-20 14:54:16.085881 0:01:13.044077
        flir ThermalCamera4      192.168.55.183 ThermalCamera4_S06C06R06.mp4 2023-11-20 14:53:01.627886 2023-11-20 14:54:16.113680 0:01:14.430647
       pelco  VisualCamera2       192.168.55.34  VisualCamera2_S06C06R06.mp4 2023-11-20 14:53:01.739311 2023-11-20 14:54:16.086559 0:01:14.312729
        

In [15]:
# ── D3. V2XHubSensor — correctly parsed, deep inspect ─────────────────────
print('=' * 68)
print('  D3. V2XHubSensor_Run_XXXX.csv — correctly parsed')
print('=' * 68)

for run in selected:
    df = v2x_sensor_parsed.get(run.name)
    if df is None:
        print(f'  {run.name}: not present')
        continue

    divider(f'{run.name}  ({len(df)} rows)')
    print(df.head(10).to_string(index=False))

    # Interpret timestamp_ms
    ts_ms = pd.to_numeric(df['timestamp_ms'], errors='coerce')
    ts_s  = ts_ms / 1000.0
    df    = df.copy()
    df['timestamp_utc'] = pd.to_datetime(ts_s, unit='s', utc=True)

    divider('Timestamp decoded')
    print(f'  Unit   : milliseconds (divide by 1000 for Unix seconds)')
    print(f'  Range  : {df["timestamp_utc"].min()}  →  {df["timestamp_utc"].max()}')
    print(f'  Rate   : ~1 row/second (V2X hub publishes 1 Hz snapshots)')

    divider('active_signal_ids analysis')
    print('  All unique values:')
    print(f'  {df["active_signal_ids"].unique().tolist()}')
    print('  Interpretation: list of currently-active signal phase IDs')
    print('  (integer IDs correspond to signal controller phase numbers)')

    divider('event_list analysis')
    print(f'  All unique values: {df["event_list"].unique().tolist()}')
    print('  Interpretation: SPaT events that fired during that second')
    print('  (empty list = no phase-change events)')

  D3. V2XHubSensor_Run_XXXX.csv — correctly parsed
  Run_1003: not present

  ── Run_1023  (65 rows) ─────────────────────────────────────────────
 timestamp_ms active_signal_ids event_list
1708650925643            [2, 8]         []
1708650926643            [2, 8]         []
1708650927643            [2, 8]         []
1708650928642            [2, 8]         []
1708650929642            [2, 8]         []
1708650930642            [2, 8]         []
1708650931642            [2, 8]         []
1708650932641            [2, 8]         []
1708650933641            [2, 8]         []
1708650934641            [2, 8]         []

  ── Timestamp decoded ───────────────────────────────────────────────
  Unit   : milliseconds (divide by 1000 for Unix seconds)
  Range  : 2024-02-23 01:15:25.642999887+00:00  →  2024-02-23 01:16:29.637000084+00:00
  Rate   : ~1 row/second (V2X hub publishes 1 Hz snapshots)

  ── active_signal_ids analysis ──────────────────────────────────────
  All unique values:
  ['[2, 8]

In [17]:
# ── D4. Radar sensor JSON — schema definition + lightweight sample table ───
print('=' * 68)
print('  D4. Radars_Run_XXXX_sensorN.json — nested schema')
print('=' * 68)

print("""
  Schema:
  List[ Record ]
    Record.topic          str   e.g. "sensor-traffic-objects/sensor1"
    Record.qos            str   e.g. "AT_LEAST_ONCE"
    Record.receivedAt     str   e.g. "2024-02-22 20:15:26"  (local time)
    Record.retain         bool
    Record.payload
      .timestamp          str   ISO 8601 UTC  e.g. "2024-02-23T01:22:59+00:00"
      .cycle_time         float seconds since last cycle (~0.1 s)
      .reference_name     str   e.g. "sensor1"
      .objects            List[ Object ]
          Object.id                  int    tracking ID
          Object.class               str    e.g. "car", "truck", "pedestrian"
          Object.heading             float  degrees
          Object.speed               float  m/s
          Object.length              float  m
          Object.acceleration        float  m/s2
          Object.position_facing     [x,y,z]  metres from radar
          Object.position_front      [x,y,z]
          Object.closest_lane        str    e.g. "lane10"
          Object.within_zone         List[str]  e.g. ["zoneAI"]
          Object.tracking_status
              .quality               float  0-1
              .new_object            bool
              .mileage               float  m total tracked
              .cycles_since_last_update int

  Note: two topic types appear in the same file:
    'sensor-traffic-objects/sensorN'  → payload has .objects (the useful data)
    'sensor-diagnostics/sensorN'      → payload is a diagnostics dict (skip for EDA)
""")

radar_flat = {}

for run in selected:
    files = find_files(run, [r'^Radars_Run_\d+_sensor1\.json$'])
    if not files:
        print(f'  {run.name}: sensor1.json not found')
        continue

    fpath = files[0]
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        data = json.load(f)

    # Filter to traffic-object records only (skip diagnostics)
    obj_records = [r for r in data
                   if isinstance(r, dict)
                   and 'traffic-objects' in r.get('topic', '')]

    divider(f'{run.name}  —  {len(data)} total records, {len(obj_records)} object records')

    if not obj_records:
        print(f'  [WARN] No traffic-object records found in {fpath.name}')
        continue

    # Flatten up to first 30 OBJECT records (not first 30 raw records)
    # This avoids the empty-DataFrame error when early records are all diagnostics
    rows = []
    for rec in obj_records[:30]:
        ts  = rec.get('payload', {}).get('timestamp', '')
        ref = rec.get('payload', {}).get('reference_name', '')
        for obj in rec.get('payload', {}).get('objects', []):
            rows.append({
                'timestamp_utc' : ts,
                'sensor'        : ref,
                'object_id'     : obj.get('id'),
                'class'         : obj.get('class'),
                'speed_ms'      : obj.get('speed'),
                'heading_deg'   : obj.get('heading'),
                'length_m'      : obj.get('length'),
                'closest_lane'  : obj.get('closest_lane'),
                'within_zone'   : str(obj.get('within_zone', [])),
                'quality'       : obj.get('tracking_status', {}).get('quality'),
            })

    flat = pd.DataFrame(rows)
    radar_flat[run.name] = flat

    if flat.empty:
        print('  [INFO] No objects found in first 30 object records (all objects lists empty).')
        continue

    print(f'  Sample flat table ({len(flat)} rows from first 30 object records):')
    print(flat.head(8).to_string(index=False))

    if 'class' in flat.columns:
        print(f'\n  Unique classes: {flat["class"].dropna().unique().tolist()}')
    if 'closest_lane' in flat.columns:
        print(f'  Unique lanes  : {sorted(flat["closest_lane"].dropna().unique().tolist())[:10]}')

  D4. Radars_Run_XXXX_sensorN.json — nested schema

  Schema:
  List[ Record ]
    Record.topic          str   e.g. "sensor-traffic-objects/sensor1"
    Record.qos            str   e.g. "AT_LEAST_ONCE"
    Record.receivedAt     str   e.g. "2024-02-22 20:15:26"  (local time)
    Record.retain         bool
    Record.payload
      .timestamp          str   ISO 8601 UTC  e.g. "2024-02-23T01:22:59+00:00"
      .cycle_time         float seconds since last cycle (~0.1 s)
      .reference_name     str   e.g. "sensor1"
      .objects            List[ Object ]
          Object.id                  int    tracking ID
          Object.class               str    e.g. "car", "truck", "pedestrian"
          Object.heading             float  degrees
          Object.speed               float  m/s
          Object.length              float  m
          Object.acceleration        float  m/s2
          Object.position_facing     [x,y,z]  metres from radar
          Object.position_front      [x,y,z]
    

In [18]:
# ── D5. traffic-triggers-output JSON — schema + lightweight event table ────
print('=' * 68)
print('  D5. traffic-triggers-output.json — nested schema')
print('=' * 68)

print("""
  Schema:
  List[ Record ]
    Record.topic           str   "traffic-triggers-output"
    Record.payload
      .timestamp           str   ISO 8601 UTC  e.g. "2024-02-23T01:22:59+00:00"
      .trigger_outputs     List[ TriggerOutput ]
          TriggerOutput.traffic_triggers  List[ Trigger ]
              Trigger.reference_name      str  e.g. "trigger_11_3"
              Trigger.associated_lane     str  e.g. "lane22"
              Trigger.associated_sensor   str  e.g. "sensor3"
              Trigger.associated_zone     str  e.g. "zoneK"

  Interpretation:
  Each record = one snapshot in time. trigger_outputs lists ALL active triggers.
  Each trigger = a vehicle (or object) occupying a zone/lane detected by a sensor.
  This is the key file for linking sensor detections to intersection lanes/zones.
""")

trigger_flat = {}

for run in selected:
    files = find_files(run, [r'traffic-triggers-output\.json$'])
    if not files:
        print(f'  {run.name}: not found')
        continue

    fpath = files[0]
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        data = json.load(f)

    divider(f'{run.name}  —  {len(data)} records  ({fmt_size(fpath)})')

    # Flatten first 50 records → one row per trigger per timestamp
    rows = []
    for rec in data[:50]:
        if not isinstance(rec, dict):
            continue
        ts = rec.get('payload', {}).get('timestamp', '')
        for to in rec.get('payload', {}).get('trigger_outputs', []):
            for trig in to.get('traffic_triggers', []):
                rows.append({
                    'timestamp_utc'    : ts,
                    'reference_name'   : trig.get('reference_name'),
                    'associated_lane'  : trig.get('associated_lane'),
                    'associated_sensor': trig.get('associated_sensor'),
                    'associated_zone'  : trig.get('associated_zone'),
                })

    flat = pd.DataFrame(rows)
    trigger_flat[run.name] = flat

    if flat.empty:
        print('  No trigger rows extracted from first 50 records.')
        continue

    print(f'  Sample event table ({len(flat)} trigger rows from first 50 records):')
    print(flat.head(10).to_string(index=False))

    divider('Unique values')
    print(f'  lanes  : {sorted(flat["associated_lane"].dropna().unique())}')
    print(f'  sensors: {sorted(flat["associated_sensor"].dropna().unique())}')
    print(f'  zones  : {sorted(flat["associated_zone"].dropna().unique())}')

    divider('Timestamp range')
    ts_parsed = pd.to_datetime(flat['timestamp_utc'], utc=True, errors='coerce').dropna()
    if not ts_parsed.empty:
        print(f'  min: {ts_parsed.min()}')
        print(f'  max: {ts_parsed.max()}')
        print(f'  (UTC confirmed — matches radar sensor payload timestamps)')

  D5. traffic-triggers-output.json — nested schema

  Schema:
  List[ Record ]
    Record.topic           str   "traffic-triggers-output"
    Record.payload
      .timestamp           str   ISO 8601 UTC  e.g. "2024-02-23T01:22:59+00:00"
      .trigger_outputs     List[ TriggerOutput ]
          TriggerOutput.traffic_triggers  List[ Trigger ]
              Trigger.reference_name      str  e.g. "trigger_11_3"
              Trigger.associated_lane     str  e.g. "lane22"
              Trigger.associated_sensor   str  e.g. "sensor3"
              Trigger.associated_zone     str  e.g. "zoneK"

  Interpretation:
  Each record = one snapshot in time. trigger_outputs lists ALL active triggers.
  Each trigger = a vehicle (or object) occupying a zone/lane detected by a sensor.
  This is the key file for linking sensor detections to intersection lanes/zones.

  Run_1003: not found

  ── Run_1023  —  2525 records  (6.27 MB) ────────────────────────────
  Sample event table (600 trigger rows from fi

---
## E. Data Dictionary

One authoritative row per file type — derived from everything above.

In [19]:
# ── E. Build data dictionary table ────────────────────────────────────────
# One row per file type. All fields confirmed from real data.

DATA_DICT = [
    {
        'file_type'         : 'GT CSV',
        'file_pattern'      : 'Run_XXXX_GT.csv',
        'example_file'      : 'Run_1003_GT.csv',
        'parsed_correctly'  : 'yes',
        'has_header'        : 'yes',
        'delimiter'         : ',',
        'row_unit'          : 'timestamp (one frame ~10 Hz)',
        'format_class'      : 'wide',
        'timestamp_field'   : 'Time',
        'timestamp_format'  : 'Unix epoch float (seconds, UTC)',
        'key_fields'        : 'Time; CLASS_xctr/yctr/zctr/xlen/ylen/zlen/xrot/yrot/zrot',
        'n_rows_typical'    : '636-758',
        'n_cols_typical'    : 'variable (depends on object classes present)',
        'special_handling'  : 'melt CLASS_FIELD columns; handle .1/.2 duplicate instances',
        'notes'             : 'Wide format. Most cells are NaN (only occupied classes non-NaN per row).',
    },
    {
        'file_type'         : 'ISC all-timing',
        'file_pattern'      : 'ISC_Run_XXXX_ISC_all_timing.csv',
        'example_file'      : 'ISC_Run_1023_ISC_all_timing.csv',
        'parsed_correctly'  : 'NO — needs header=None',
        'has_header'        : 'NO (headerless)',
        'delimiter'         : ',',
        'row_unit'          : 'one sensor stream metadata entry',
        'format_class'      : 'event_log',
        'timestamp_field'   : 'start_time, end_time',
        'timestamp_format'  : 'datetime string YYYY-MM-DD HH:MM:SS.ffffff (local time)',
        'key_fields'        : 'sensor_type, sensor_name, ip_address, filename, start_time, end_time, duration',
        'n_rows_typical'    : '13-14',
        'n_cols_typical'    : '7',
        'special_handling'  : 'read with header=None; assign 7 column names; parse start/end as datetime',
        'notes'             : 'Use to verify which sensors were active and their recording windows.',
    },
    {
        'file_type'         : 'V2X hub timing',
        'file_pattern'      : 'ISC_Run_XXXX_v2xhub_timing.csv',
        'example_file'      : 'ISC_Run_1023_v2xhub_timing.csv',
        'parsed_correctly'  : 'NO — needs header=None',
        'has_header'        : 'NO (headerless)',
        'delimiter'         : ',',
        'row_unit'          : 'one V2X stream metadata entry',
        'format_class'      : 'event_log',
        'timestamp_field'   : 'start_time, end_time',
        'timestamp_format'  : 'datetime string YYYY-MM-DD HH:MM:SS.ffffff (local time)',
        'key_fields'        : 'sensor_type, sensor_name, ip_address, filename, start_time, end_time, duration',
        'n_rows_typical'    : '2',
        'n_cols_typical'    : '7',
        'special_handling'  : 'same as ISC_all_timing; only present in V2X-equipped runs',
        'notes'             : 'Rows: v2xhub_sensors (BSM/CSV) and v2xhub_spat (SPaT/PCAP).',
    },
    {
        'file_type'         : 'Visual camera frame-timing',
        'file_pattern'      : 'VisualCameraX_Run_XXXX_frame-timing.csv',
        'example_file'      : 'VisualCamera1_Run_1003_frame-timing.csv',
        'parsed_correctly'  : 'yes',
        'has_header'        : 'yes',
        'delimiter'         : ',',
        'row_unit'          : 'one video frame',
        'format_class'      : 'long',
        'timestamp_field'   : 'Timestamp',
        'timestamp_format'  : 'ISC: YYYY-MM-DD-HH-MM-SS_microseconds (LOCAL time)',
        'key_fields'        : 'Image_number, Timestamp',
        'n_rows_typical'    : '1670-4060 (varies per camera/run)',
        'n_cols_typical'    : '2',
        'special_handling'  : 'parse ISC timestamp with custom parser; add UTC offset for alignment',
        'notes'             : '8 cameras/run. ~30 fps. Image_number = 0-based frame index into .mp4.',
    },
    {
        'file_type'         : 'Thermal camera frame-timing',
        'file_pattern'      : 'ThermalCameraX_Run_XXXX_frame-timing.csv',
        'example_file'      : 'ThermalCamera1_Run_1003_frame-timing.csv',
        'parsed_correctly'  : 'yes',
        'has_header'        : 'yes',
        'delimiter'         : ',',
        'row_unit'          : 'one thermal frame',
        'format_class'      : 'long',
        'timestamp_field'   : 'Timestamp',
        'timestamp_format'  : 'ISC: YYYY-MM-DD-HH-MM-SS_microseconds (LOCAL time)',
        'key_fields'        : 'Image_number, Timestamp',
        'n_rows_typical'    : '1875-2249',
        'n_cols_typical'    : '2',
        'special_handling'  : 'same as visual; 5 cameras/run',
        'notes'             : 'Identical schema to visual timing. Thermal cameras run at same ~30 fps.',
    },
    {
        'file_type'         : 'V2X hub sensor',
        'file_pattern'      : 'V2XHubSensor_Run_XXXX.csv',
        'example_file'      : 'V2XHubSensor_Run_1023.csv',
        'parsed_correctly'  : 'NO — needs header=None',
        'has_header'        : 'NO (headerless)',
        'delimiter'         : ',',
        'row_unit'          : '1-second V2X snapshot',
        'format_class'      : 'event_log',
        'timestamp_field'   : 'timestamp_ms',
        'timestamp_format'  : 'Unix epoch milliseconds (integer)',
        'key_fields'        : 'timestamp_ms, active_signal_ids, event_list',
        'n_rows_typical'    : '~70',
        'n_cols_typical'    : '3',
        'special_handling'  : 'header=None; 3 cols; divide ts by 1000 for seconds; parse list cols',
        'notes'             : 'active_signal_ids = currently active signal phases. event_list = phase-change events.',
    },
    {
        'file_type'         : 'Radar sensor JSON',
        'file_pattern'      : 'Radars_Run_XXXX_sensorN.json',
        'example_file'      : 'Radars_Run_1023_sensor1.json',
        'parsed_correctly'  : 'yes (with custom extractor)',
        'has_header'        : 'N/A (JSON)',
        'delimiter'         : 'N/A',
        'row_unit'          : 'one MQTT cycle (~0.1 s)',
        'format_class'      : 'nested_json',
        'timestamp_field'   : 'payload.timestamp',
        'timestamp_format'  : 'ISO 8601 UTC: YYYY-MM-DDTHH:MM:SS+00:00',
        'key_fields'        : 'topic, payload.timestamp, payload.objects[id/class/speed/lane/zone]',
        'n_rows_typical'    : '~1900 records/sensor; 4 sensors/run',
        'n_cols_typical'    : 'N/A',
        'special_handling'  : 'filter topic=traffic-objects; iterate payload.objects to flatten',
        'notes'             : 'Two topic types: traffic-objects (use) and diagnostics (skip). ~1-2 MB/sensor.',
    },
    {
        'file_type'         : 'Traffic triggers JSON',
        'file_pattern'      : 'Radars_Run_XXXX_traffic-triggers-output.json',
        'example_file'      : 'Radars_Run_1023_traffic-triggers-output.json',
        'parsed_correctly'  : 'yes (with custom extractor)',
        'has_header'        : 'N/A (JSON)',
        'delimiter'         : 'N/A',
        'row_unit'          : 'one MQTT message per timestamp',
        'format_class'      : 'nested_json',
        'timestamp_field'   : 'payload.timestamp',
        'timestamp_format'  : 'ISO 8601 UTC: YYYY-MM-DDTHH:MM:SS+00:00',
        'key_fields'        : 'payload.timestamp, trigger.associated_lane, trigger.associated_sensor, trigger.associated_zone',
        'n_rows_typical'    : '~2525 messages; variable triggers/message',
        'n_cols_typical'    : 'N/A',
        'special_handling'  : 'iterate trigger_outputs[].traffic_triggers[] to flatten per-trigger rows',
        'notes'             : 'Most informative file. Links sensor detections to intersection zones/lanes. 6-9 MB.',
    },
]

dict_df = pd.DataFrame(DATA_DICT)

print('Data Dictionary\n')
pd.set_option('display.max_colwidth', 70)
display_cols = ['file_type','file_pattern','parsed_correctly','has_header',
                'delimiter','row_unit','timestamp_field','timestamp_format',
                'special_handling','notes']
print(dict_df[display_cols].to_string(index=False))

Data Dictionary

                  file_type                                 file_pattern            parsed_correctly      has_header delimiter                         row_unit      timestamp_field                                        timestamp_format                                                          special_handling                                                                                 notes
                     GT CSV                              Run_XXXX_GT.csv                         yes             yes         ,     timestamp (one frame ~10 Hz)                 Time                         Unix epoch float (seconds, UTC)                melt CLASS_FIELD columns; handle .1/.2 duplicate instances              Wide format. Most cells are NaN (only occupied classes non-NaN per row).
             ISC all-timing              ISC_Run_XXXX_ISC_all_timing.csv      NO — needs header=None NO (headerless)         , one sensor stream metadata entry start_time, end_time datetime

---
## F. Save Output

In [20]:
# ── F. Save data dictionary to CSV ────────────────────────────────────────
out_path = OUT_DIR / 'parser_data_dictionary.csv'
dict_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {dict_df.shape}')

# Print the parsing verdict summary
print('\n' + '='*68)
print('  PARSING VERDICT SUMMARY')
print('='*68)
for _, row in dict_df.iterrows():
    verdict = row['parsed_correctly']
    icon    = 'OK' if verdict.startswith('yes') else '!!'
    print(f'  [{icon}]  {row["file_pattern"]}')
    if icon == '!!':
        print(f'        FIX: {row["special_handling"]}')

print('\n' + '='*68)
print('  KEY TIMEZONE NOTE')
print('='*68)
print('  GT timestamps       : UTC (Unix epoch)')
print('  Camera timestamps   : LOCAL time (observed 5 h behind UTC in runs 1003/1023/1027)')
print('  Radar JSON timestamps: UTC (ISO 8601 +00:00)')
print('  V2X timestamps      : UTC (Unix epoch milliseconds)')
print('  ISC timing file     : local datetime strings (same offset as camera)')
print()
print('  ACTION: determine site UTC offset once and apply to camera/ISC timing files.')

Saved: /content/outputs/tables/parser_data_dictionary.csv
Shape: (8, 15)

  PARSING VERDICT SUMMARY
  [OK]  Run_XXXX_GT.csv
  [!!]  ISC_Run_XXXX_ISC_all_timing.csv
        FIX: read with header=None; assign 7 column names; parse start/end as datetime
  [!!]  ISC_Run_XXXX_v2xhub_timing.csv
        FIX: same as ISC_all_timing; only present in V2X-equipped runs
  [OK]  VisualCameraX_Run_XXXX_frame-timing.csv
  [OK]  ThermalCameraX_Run_XXXX_frame-timing.csv
  [!!]  V2XHubSensor_Run_XXXX.csv
        FIX: header=None; 3 cols; divide ts by 1000 for seconds; parse list cols
  [OK]  Radars_Run_XXXX_sensorN.json
  [OK]  Radars_Run_XXXX_traffic-triggers-output.json

  KEY TIMEZONE NOTE
  GT timestamps       : UTC (Unix epoch)
  Camera timestamps   : LOCAL time (observed 5 h behind UTC in runs 1003/1023/1027)
  Radar JSON timestamps: UTC (ISO 8601 +00:00)
  V2X timestamps      : UTC (Unix epoch milliseconds)
  ISC timing file     : local datetime strings (same offset as camera)

  ACTION: determin